---

## NOTEBOOK 9 — Agent IA (classification + recommandation)

---

### Objectif

Construire deux agents de classification de genre musical :

- **Agent V1 (baseline)** : XGBoost audio (351D) + recommandation cosine CNN
- **Agent V2 (full)** : ensemble audio + NLP + PANNs + recommandation
  + SHAP explicabilite + detection atypiques + Claude API

---

### Plan du notebook

| Cellule | Section | Contenu |
|---------|---------|--------|
| C2 | 1. Configuration | Imports, chemins, chargement donnees |
| C3 | 2. Agent V1 | XGBoost audio seul (features_V2 351D) |
| C4 | 3. Agent V2 | Ensemble audio + NLP + reco + SHAP |
| C5 | 4. Demo | Test sur 5 pistes aleatoires |
| C6 | 5. Evaluation | Comparaison V1 vs V2 sur test set complet |
| C7 | 6. Export | Sauvegarde modeles pour Streamlit |
| --- | Conclusion | Synthese, performances, usage |

---


In [5]:
# C2
# 1. Configuration

import numpy as np
import pandas as pd
import warnings
import time
import re
from pathlib import Path
from html import unescape

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

SEED = 42
TEST_SIZE = 0.2
BASE = Path.cwd()
OUTPUT_DIR = BASE / 'outputs'
FEATURES_CSV = OUTPUT_DIR / 'features' / 'features_V2.csv'
CNN_DIR = OUTPUT_DIR / 'cnn'
META_DIR = BASE / 'data' / 'raw' / 'fma_metadata'
AGENT_DIR = OUTPUT_DIR / 'agent'
AGENT_DIR.mkdir(parents=True, exist_ok=True)

# --- Chargement features audio ---
df_feat = pd.read_csv(FEATURES_CSV)
df_feat = df_feat.loc[:, ~df_feat.columns.astype(str).str.startswith('Unnamed:')]

LABEL_COLS = [
    'track_id', 'track_id_int', 'genre_top', 'artist_name', 'genres_decoded',
    'genres', 'n_subgenres', 'mismatch', 'mismatch_calc', 'track_title',
    'year', 'duration', 'bit_rate'
]
FEATURE_COLS = [c for c in df_feat.columns if c not in LABEL_COLS]

X_audio = df_feat[FEATURE_COLS]
y = df_feat['genre_top'].astype(str)
groups = df_feat['artist_name'].astype(str)

le = LabelEncoder()
y_enc = le.fit_transform(y)
genres = list(le.classes_)

# --- Split ---
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(gss.split(X_audio, y_enc, groups=groups))

print('Features audio :', X_audio.shape)
print('Train :', len(train_idx), '| Test :', len(test_idx))
print('Genres :', genres)

# --- Chargement NLP ---
tracks = pd.read_csv(META_DIR / 'tracks.csv', index_col=0, header=[0, 1])
small = tracks[tracks[('set', 'subset')] == 'small'].copy()
feat_ids = set(df_feat['track_id'].values)
small = small[small.index.isin(feat_ids)]

def clean_html(text):
    if pd.isna(text) or text == '':
        return ''
    text = unescape(str(text))
    text = re.sub(r'<[^>]+>', ' ', text)
    return re.sub(r'\\s+', ' ', text).strip()

df_nlp = pd.DataFrame({
    'track_id': small.index,
    'text': (
        small[('track', 'title')].fillna('').astype(str) + ' ' +
        small[('album', 'title')].fillna('').astype(str) + ' ' +
        small[('artist', 'bio')].fillna('').apply(clean_html)
    ).values,
}).reset_index(drop=True)

text_map = dict(zip(df_nlp['track_id'], df_nlp['text']))
df_feat['text'] = df_feat['track_id'].map(text_map).fillna('')

# --- Chargement embeddings CNN ---
embeddings_test = np.load(CNN_DIR / 'embeddings_test.npy')
test_track_ids_cnn = np.load(CNN_DIR / 'test_track_ids.npy')
test_genres_cnn = np.load(CNN_DIR / 'test_genres.npy', allow_pickle=True)

print('NLP textes :', (df_feat['text'] != '').sum(), '/', len(df_feat))
print('Embeddings CNN :', embeddings_test.shape)
print('Config OK')

# --- Chargement PANNs embeddings (NB7) ---
PANNS_DIR = OUTPUT_DIR / 'transfer_learning'
PANNS_PKL = PANNS_DIR / 'embeddings_panns.pkl'
panns_available = False
if PANNS_PKL.exists():
    import pickle
    with open(PANNS_PKL, 'rb') as f:
        panns_dict = pickle.load(f)
    print('PANNs embeddings :', len(panns_dict), 'pistes')
    panns_available = True
else:
    print('PANNs embeddings non disponibles (NB7 pas run)')
    panns_dict = {}


Features audio : (7994, 351)
Train : 6477 | Test : 1517
Genres : ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']
NLP textes : 7994 / 7994
Embeddings CNN : (1517, 256)
Config OK
PANNs embeddings : 7994 pistes


In [6]:
# C3
# 2. Agent V1 -- XGBoost audio seul

print('=== ENTRAINEMENT AGENT V1 ===')
print('Modele : XGBoost sur features audio (351D)')
print()

sw = compute_sample_weight('balanced', y_enc[train_idx])
pipe_v1 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc', RobustScaler()),
    ('clf', XGBClassifier(
        n_estimators=300, max_depth=20, min_child_weight=8,
        random_state=SEED, eval_metric='mlogloss', verbosity=0, n_jobs=-1
    ))
])

t0 = time.time()
pipe_v1.fit(X_audio.iloc[train_idx], y_enc[train_idx], clf__sample_weight=sw)
dur_v1 = time.time() - t0

y_pred_v1 = pipe_v1.predict(X_audio.iloc[test_idx])
f1_v1 = f1_score(y_enc[test_idx], y_pred_v1, average='macro')
acc_v1 = accuracy_score(y_enc[test_idx], y_pred_v1)

print('Entraine en', round(dur_v1, 1), 's')
print('F1 macro :', round(f1_v1, 4))
print('Accuracy :', round(acc_v1, 4))

def agent_v1_predict(features_row):
    pred_enc = pipe_v1.predict(features_row.values.reshape(1, -1))[0]
    proba = pipe_v1.predict_proba(features_row.values.reshape(1, -1))[0]
    genre = le.inverse_transform([pred_enc])[0]
    confidence = proba.max() * 100
    top3 = [(le.inverse_transform([i])[0], round(p * 100, 1))
            for i, p in sorted(enumerate(proba), key=lambda x: -x[1])[:3]]
    return {'genre': genre, 'confidence': round(confidence, 1), 'top3': top3, 'agent': 'V1'}

print()
print('Agent V1 pret')


=== ENTRAINEMENT AGENT V1 ===
Modele : XGBoost sur features audio (351D)

Entraine en 13.0 s
F1 macro : 0.4907
Accuracy : 0.501

Agent V1 pret


In [7]:
# C4
# 3. Agent V2 -- Ensemble audio + NLP + reco + SHAP

print('=== ENTRAINEMENT AGENT V2 ===')
print('Modeles : XGBoost audio + LogReg NLP')
print()

# --- NLP : TF-IDF + LogReg ---
tfidf = TfidfVectorizer(
    max_features=10000, ngram_range=(1, 2),
    min_df=3, max_df=0.95, sublinear_tf=True, strip_accents='unicode'
)

X_text_train = tfidf.fit_transform(df_feat.iloc[train_idx]['text'].values)
X_text_test = tfidf.transform(df_feat.iloc[test_idx]['text'].values)

lr_nlp = LogisticRegression(max_iter=1000, C=10.0, random_state=SEED, n_jobs=-1)
lr_nlp.fit(X_text_train, y_enc[train_idx])

f1_nlp = f1_score(y_enc[test_idx], lr_nlp.predict(X_text_test), average='macro')
print('LogReg NLP entraine | F1 =', round(f1_nlp, 4))

# --- Poids ensemble ---
w_audio = f1_v1
w_nlp = f1_nlp
w_total = w_audio + w_nlp
w_audio_norm = w_audio / w_total
w_nlp_norm = w_nlp / w_total
print('Poids ensemble : audio=' + str(round(w_audio_norm, 2)) + ' | nlp=' + str(round(w_nlp_norm, 2)))


# --- PANNs : XGBoost sur embeddings 2048D (si disponible) ---
panns_xgb = None
f1_panns = 0
w_panns_norm = 0

if panns_available:
    # Construire X_panns aligne avec df_feat
    panns_ids = []
    panns_vecs = []
    for _, row_p in df_feat.iterrows():
        tid = int(row_p['track_id'])
        if tid in panns_dict:
            panns_ids.append(tid)
            panns_vecs.append(panns_dict[tid])
    
    if len(panns_vecs) > 100:
        X_panns_full = np.array(panns_vecs)
        panns_id_to_idx = {tid: i for i, tid in enumerate(panns_ids)}
        
        # Indices train/test parmi les pistes qui ont des embeddings PANNs
        train_panns = [i for i, idx in enumerate(panns_ids) if idx in set(df_feat.iloc[train_idx]['track_id'])]
        test_panns = [i for i, idx in enumerate(panns_ids) if idx in set(df_feat.iloc[test_idx]['track_id'])]
        
        if len(train_panns) > 50 and len(test_panns) > 50:
            y_panns_train = le.transform(df_feat.set_index('track_id').loc[[panns_ids[i] for i in train_panns], 'genre_top'])
            y_panns_test = le.transform(df_feat.set_index('track_id').loc[[panns_ids[i] for i in test_panns], 'genre_top'])
            
            sw_p = compute_sample_weight('balanced', y_panns_train)
            panns_xgb = Pipeline([
                ('imp', SimpleImputer(strategy='median')),
                ('sc', RobustScaler()),
                ('clf', XGBClassifier(n_estimators=100, max_depth=10, min_child_weight=4,
                                      random_state=SEED, eval_metric='mlogloss', verbosity=0, n_jobs=-1))
            ])
            panns_xgb.fit(X_panns_full[train_panns], y_panns_train, clf__sample_weight=sw_p)
            f1_panns = f1_score(y_panns_test, panns_xgb.predict(X_panns_full[test_panns]), average='macro')
            print('XGBoost PANNs entraine | F1 =', round(f1_panns, 4))
        else:
            print('Pas assez de pistes PANNs pour entrainer')
    else:
        print('Pas assez embeddings PANNs')

# --- Recalcul poids ensemble (audio + nlp + panns) ---
w_audio = f1_v1
w_nlp = f1_nlp
w_panns = f1_panns
w_total = w_audio + w_nlp + w_panns
if w_total > 0:
    w_audio_norm = w_audio / w_total
    w_nlp_norm = w_nlp / w_total
    w_panns_norm = w_panns / w_total
print('Poids V2 : audio=' + str(round(w_audio_norm, 3)) + ' nlp=' + str(round(w_nlp_norm, 3)) + ' panns=' + str(round(w_panns_norm, 3)))

# --- SHAP ---
import shap
explainer = shap.TreeExplainer(pipe_v1.named_steps['clf'])
preprocessor_v1 = Pipeline(pipe_v1.steps[:-1])
print('SHAP explainer pret')

# --- Embeddings CNN ---
emb_map = dict(zip(test_track_ids_cnn, range(len(test_track_ids_cnn))))

def agent_v2_predict(features_row, text='', track_id=None):
    X_row = features_row.values.reshape(1, -1)
    proba_audio = pipe_v1.predict_proba(X_row)[0]
    X_text = tfidf.transform([text])
    proba_nlp = lr_nlp.predict_proba(X_text)[0]
    proba_ensemble = w_audio_norm * proba_audio + w_nlp_norm * proba_nlp
    # PANNs (si disponible pour cette piste)
    if panns_xgb is not None and track_id is not None and track_id in panns_dict:
        X_panns_row = np.array(panns_dict[track_id]).reshape(1, -1)
        proba_panns = panns_xgb.predict_proba(X_panns_row)[0]
        proba_ensemble = w_audio_norm * proba_audio + w_nlp_norm * proba_nlp + w_panns_norm * proba_panns
        result['proba_panns'] = le.inverse_transform([proba_panns.argmax()])[0]
    pred_enc = proba_ensemble.argmax()
    genre = le.inverse_transform([pred_enc])[0]
    confidence = proba_ensemble.max() * 100
    top3 = [(le.inverse_transform([i])[0], round(p * 100, 1))
            for i, p in sorted(enumerate(proba_ensemble), key=lambda x: -x[1])[:3]]
    result = {
        'genre': genre, 'confidence': round(confidence, 1), 'top3': top3, 'agent': 'V2',
        'proba_audio': le.inverse_transform([proba_audio.argmax()])[0],
        'proba_nlp': le.inverse_transform([proba_nlp.argmax()])[0],
    }
    # SHAP
    X_proc = preprocessor_v1.transform(X_row)
    sv = explainer.shap_values(X_proc)
    if isinstance(sv, np.ndarray) and sv.ndim == 3:
        shap_for_pred = sv[0, :, pred_enc]
    else:
        shap_for_pred = sv[pred_enc][0]
    top5_shap_idx = np.argsort(np.abs(shap_for_pred))[::-1][:5]
    result['shap_top5'] = [(FEATURE_COLS[j], round(float(shap_for_pred[j]), 4)) for j in top5_shap_idx]
    # Recommandation
    if track_id is not None and track_id in emb_map:
        idx = emb_map[track_id]
        seed_emb = embeddings_test[idx].reshape(1, -1)
        sims = cosine_similarity(seed_emb, embeddings_test)[0]
        top5_reco = sims.argsort()[::-1][1:6]
        result['reco'] = []
        for ri in top5_reco:
            tid = test_track_ids_cnn[ri]
            row_meta = df_feat[df_feat['track_id'] == tid]
            title_r = row_meta['track_title'].values[0] if len(row_meta) > 0 and 'track_title' in row_meta.columns else str(tid)
            artist_r = row_meta['artist_name'].values[0] if len(row_meta) > 0 else '?'
            result['reco'].append({
                'track_id': int(tid), 'title': title_r, 'artist': artist_r,
                'genre': test_genres_cnn[ri], 'similarity': round(sims[ri] * 100, 1)
            })
    return result

print()
print('Agent V2 pret')


=== ENTRAINEMENT AGENT V2 ===
Modeles : XGBoost audio + LogReg NLP

LogReg NLP entraine | F1 = 0.4599
Poids ensemble : audio=0.52 | nlp=0.48
XGBoost PANNs entraine | F1 = 0.5873
Poids V2 : audio=0.319 nlp=0.299 panns=0.382
SHAP explainer pret

Agent V2 pret


In [8]:
# C5
# 4. Export -- sauvegarde modeles pour Streamlit

import joblib

joblib.dump(pipe_v1, AGENT_DIR / 'pipe_xgb_audio.joblib')
joblib.dump(lr_nlp, AGENT_DIR / 'lr_nlp.joblib')
joblib.dump(tfidf, AGENT_DIR / 'tfidf_vectorizer.joblib')
joblib.dump(le, AGENT_DIR / 'label_encoder.joblib')
joblib.dump(preprocessor_v1, AGENT_DIR / 'preprocessor_v1.joblib')
np.save(AGENT_DIR / 'feature_cols.npy', np.array(FEATURE_COLS))
np.save(AGENT_DIR / 'weights.npy', np.array([w_audio_norm, w_nlp_norm]))

# Metadata pour Streamlit
df_meta = df_feat[['track_id', 'genre_top', 'artist_name', 'track_title', 'text']].copy()
if 'mismatch' in df_feat.columns:
    df_meta['mismatch'] = df_feat['mismatch']
df_meta.to_csv(AGENT_DIR / 'metadata.csv', index=False)

# Suspects top30 (si disponible)
suspects_path = OUTPUT_DIR / 'curation' / 'suspects_top30.csv'
if suspects_path.exists():
    import shutil
    shutil.copy(suspects_path, AGENT_DIR / 'suspects_top30.csv')
    print('suspects_top30.csv copie dans agent/')

print('Modeles sauvegardes dans', AGENT_DIR)
print()
for f in sorted(AGENT_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print('  ' + f.name.ljust(35) + str(round(size)) + ' KB')

print()
print('Pour lancer Streamlit :')
print('  streamlit run app_streamlit.py')

if panns_xgb is not None:
    joblib.dump(panns_xgb, AGENT_DIR / 'panns_xgb.joblib')
    print('PANNs XGBoost sauvegarde')
np.save(AGENT_DIR / 'weights_v2.npy', np.array([w_audio_norm, w_nlp_norm, w_panns_norm]))


suspects_top30.csv copie dans agent/
Modeles sauvegardes dans c:\STOCKAGE_XIA\DU SDA\MACHINE LEARNING 2\PROJET\outputs\agent

  feature_cols.npy                   41 KB
  label_encoder.joblib               1 KB
  lr_nlp.joblib                      626 KB
  metadata.csv                       4804 KB
  pipe_xgb_audio.joblib              3439 KB
  preprocessor_v1.joblib             16 KB
  suspects_top30.csv                 1 KB
  tfidf_vectorizer.joblib            382 KB
  weights.npy                        0 KB

Pour lancer Streamlit :
  streamlit run app_streamlit.py
PANNs XGBoost sauvegarde


In [9]:
# C6
# 5. Demo -- 5 pistes via agent.py

import sys
sys.path.insert(0, str(BASE / "src"))
from agent import run_agent

import random
random.seed(SEED)
demo_indices = random.sample(list(test_idx), 5)

for idx in demo_indices:
    tid = int(df_feat.iloc[idx]["track_id"])
    genre_real = df_feat.iloc[idx]["genre_top"]
    
    r1 = run_agent(track_id=tid, mode="V1")
    r2 = run_agent(track_id=tid, mode="V2")
    
    print("=" * 65)
    print("Piste :", r1["title"], "-", r1["artist"])
    print("Genre reel :", genre_real)
    print()
    
    ok1 = "OK" if r1["pred_genre"] == genre_real else "ERREUR"
    print("  Agent V1 :", r1["pred_genre"], "(" + str(round(r1["confidence"]*100)) + "%)", "[" + ok1 + "]")
    print("    Top 3  :", r1["top_k"])
    
    ok2 = "OK" if r2["pred_genre"] == genre_real else "ERREUR"
    print("  Agent V2 :", r2["pred_genre"], "(" + str(round(r2["confidence"]*100)) + "%)", "[" + ok2 + "]")
    print("    Audio  :", r2.get("pred_audio"), "| NLP :", r2.get("pred_nlp"), "| Ensemble :", r2["pred_genre"])
    if r2.get("shap_features"):
        print("    SHAP   :", r2["shap_features"][:3])
    if r2.get("reco"):
        print("    Reco   :")
        for rec in r2["reco"]:
            print("     ", rec["title"], "-", rec["artist"], "(" + rec["genre"] + ",", str(rec["similarity"]) + "%)")
    print()


Piste : Karcığar Ud taksim - Ehl-i Keyif
Genre reel : International

  Agent V1 : Folk (55%) [ERREUR]
    Top 3  : [('Folk', 0.5502338409423828), ('Instrumental', 0.3894136846065521), ('Experimental', 0.02620508149266243)]
  Agent V2 : Instrumental (32%) [ERREUR]
    Audio  : Folk | NLP : International | Ensemble : Instrumental
    SHAP   : [(np.str_('mfcc_07_std'), 0.3736), (np.str_('tonnetz_05_std'), 0.3123), (np.str_('spectral_centroid_01_max'), 0.2546)]
    Reco   :
      Live Improv - Brahim Fribgane (International, 96.3%)
      -Shab Ayum - Turku, Nomads of the Silk Road (International, 96.2%)
      Deep Blue Sea (with Jean Ritchie) - Howie Mitchell (Folk, 95.1%)
      David's Tune - Howie Mitchell (Folk, 94.3%)
      Amazigh - Brahim Fribgane (International, 94.1%)

Piste : Rocket Into The Future - Cullah
Genre reel : Hip-Hop

  Agent V1 : Electronic (50%) [ERREUR]
    Top 3  : [('Electronic', 0.5032125115394592), ('Pop', 0.3077995181083679), ('International', 0.0541616044938564

In [10]:
# C7
# 6. Evaluation -- V1 vs V2 sur test set complet

print('=== EVALUATION COMPLETE ===')
print()

# V1
y_pred_v1_all = pipe_v1.predict(X_audio.iloc[test_idx])
f1_v1_final = f1_score(y_enc[test_idx], y_pred_v1_all, average='macro')

# V2 ensemble
proba_audio_all = pipe_v1.predict_proba(X_audio.iloc[test_idx])
proba_nlp_all = lr_nlp.predict_proba(X_text_test)
proba_v2_all = w_audio_norm * proba_audio_all + w_nlp_norm * proba_nlp_all
y_pred_v2_all = proba_v2_all.argmax(axis=1)
f1_v2_final = f1_score(y_enc[test_idx], y_pred_v2_all, average='macro')

print('Agent V1 (XGBoost audio seul)  : F1 =', round(f1_v1_final, 4))
print('Agent V2 (ensemble audio+NLP)  : F1 =', round(f1_v2_final, 4))
delta = f1_v2_final - f1_v1_final
print('Delta V2 - V1                  :', ('+' if delta > 0 else '') + str(round(delta, 4)))
print()

# Par genre
print('--- Agent V1 ---')
print(classification_report(y_enc[test_idx], y_pred_v1_all, target_names=genres, zero_division=0))
print('--- Agent V2 ---')
print(classification_report(y_enc[test_idx], y_pred_v2_all, target_names=genres, zero_division=0))

# Gains par genre
print('=== GAINS PAR GENRE (V2 vs V1) ===')
r1 = classification_report(y_enc[test_idx], y_pred_v1_all, target_names=genres, output_dict=True, zero_division=0)
r2 = classification_report(y_enc[test_idx], y_pred_v2_all, target_names=genres, output_dict=True, zero_division=0)
for g in genres:
    d = r2[g]['f1-score'] - r1[g]['f1-score']
    arrow = '+' if d > 0 else ''
    print('  ' + g.ljust(15) + ': V1=' + str(round(r1[g]['f1-score'], 3)).ljust(7) + ' V2=' + str(round(r2[g]['f1-score'], 3)).ljust(7) + ' delta=' + arrow + str(round(d, 3)))


=== EVALUATION COMPLETE ===

Agent V1 (XGBoost audio seul)  : F1 = 0.4907
Agent V2 (ensemble audio+NLP)  : F1 = 0.5632
Delta V2 - V1                  : +0.0724

--- Agent V1 ---
               precision    recall  f1-score   support

   Electronic       0.48      0.54      0.51       249
 Experimental       0.51      0.44      0.47       199
         Folk       0.46      0.61      0.53       119
      Hip-Hop       0.69      0.59      0.64       233
 Instrumental       0.42      0.52      0.46       170
International       0.60      0.44      0.51       249
          Pop       0.18      0.21      0.19       128
         Rock       0.63      0.60      0.61       170

     accuracy                           0.50      1517
    macro avg       0.50      0.49      0.49      1517
 weighted avg       0.52      0.50      0.51      1517

--- Agent V2 ---
               precision    recall  f1-score   support

   Electronic       0.53      0.59      0.56       249
 Experimental       0.58      0

---

## Conclusion

---

### Deux agents, deux niveaux

| | Agent V1 | Agent V2 |
|--|---------|----------|
| Modeles | XGBoost audio (351D) | XGBoost audio + LogReg NLP (ensemble) |
| Vitesse | Rapide, sans GPU | Rapide, sans GPU |
| Explicabilite | Non | SHAP top 5 features |
| Recommandation | Cosine similarity CNN 256D | Cosine similarity CNN 256D |
| Usage | Classification + decouverte | Classification + decouverte + explicabilite |

### Application Streamlit

L'app `app_streamlit.py` permet de :
- Selectionner une piste ou uploader un MP3
- Comparer les predictions V1 vs V2
- Voir l'explicabilite SHAP
- Obtenir 5 recommandations similaires
- Optionnel : explication Claude API en langage naturel

---
